# 08 同步任务、异步回调与任务控制

## 学习目标

- 对比阻塞模式和 `on_task_done` 回调模式。
- 理解 `task_id`。
- 了解暂停、恢复、取消。
- 知道回调模式下 `timeout_sec` 等的是 start 确认。

参考文献：文档第 4.2、6.6 章。

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from helpers import DEFAULT_WS_URL

WS_URL = DEFAULT_WS_URL
print("WS_URL =", WS_URL)

In [ ]:
CONNECT_ROBOT = False
RUN_SYNC_CAPTURE = False
RUN_ASYNC_CAPTURE = False
ENABLE_CANCEL = False

In [ ]:
import time
from helpers import connect_robot, disconnect_safely
from bajie_sdk import CameraType, Command

robot = connect_robot(WS_URL) if CONNECT_ROBOT else None
async_task_id = None

In [ ]:
if robot and RUN_SYNC_CAPTURE:
    view = robot.eco_captureImages(CameraType.HEAD, timeout_sec=30.0)
    print("同步拍照完成:", view.rgb_image.img.shape)
else:
    print("Sync capture disabled.")

In [ ]:
done_results = []

def on_images_ready(result):
    print("回调收到图像:", result.rgb_image.img.shape)
    done_results.append(result)

if robot and RUN_ASYNC_CAPTURE:
    async_task_id = robot.eco_captureImages(
        CameraType.HEAD,
        timeout_sec=30.0,
        on_task_done=on_images_ready,
    )
    print("立即返回 task_id:", async_task_id)
    print("等待回调几秒...")
    time.sleep(5)
    print("回调结果数量:", len(done_results))
    print("如果数量仍为 0，任务可能还没 finish，或 notebook 主线程等待时间太短。")
else:
    print("Async capture disabled.")

In [ ]:
# 任务控制需要真实 task_id。默认关闭。
if robot and ENABLE_CANCEL and async_task_id:
    st = robot.eco_missionControl(Command.CANCEL, async_task_id, timeout_sec=30.0)
    print(st)
else:
    print("Cancel disabled or no task_id.")

In [ ]:
disconnect_safely(robot)

## 机器人习惯

- 长任务适合异步，例如导航、回充、抓取。
- 回调在 SDK 收包线程执行，回调里不要做耗时操作。
- 取消/暂停/恢复需要正确的 `task_id`。
- 回调模式下，`timeout_sec` 用于等待 mission start 确认，不是等待 finish。

## 故障排查卡片

- 回调没有触发：确认任务是否真的完成，主线程是否过早退出。
- 取消失败：确认任务仍在运行、`task_id` 正确。
- 不确定当前任务：先看 `eco_robotWorkState()`。

## 小练习

用一句话说明同步调用和异步回调模式的返回值差异。